# Model: Liquid-Line Restriction (Binary Classification)

## Feature choice, informed by notebook 05's EDA

Per notebook 05: RTU_REFG_SUCT_PRES and RTU_REFG_SUCT_TEMP were the strongest,
though threshold-shaped (not smoothly monotonic) signals. RTU_REFG_DISC_PRES was
flagged as non-monotonic even after stage-2 filtering - included anyway, same
reasoning as overcharge (a nonlinear model may extract value from it despite the
non-monotonicity). Capacity's raw effect was substantially inflated by staging
noise (10-bar unfiltered -18.38% vs -7.49% filtered) - real, but much weaker than
the raw numbers suggested.

## Real, distinguishing feature of this fault vs. the first four modeled

This is the first fault modeled with a genuine THRESHOLD effect rather than a
roughly continuous severity scale - minimal signal at 1-4 bar, a large jump at
8-10 bar (Cohen's d=1.445 for that specific transition). All four prior faults
had smoothly-scaling severities. Worth watching whether this threshold shape
produces yet another distinct generalization pattern, or whether it behaves like
one of the three already seen.

In [1]:
import sys
from pathlib import Path

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from sklearn.model_selection import TimeSeriesSplit, train_test_split  # noqa: E402
from src.features.build_features import build_feature_table  # noqa: E402

table = build_feature_table(
    baseline_path="../data/raw/RTU_sim_baseline.csv",
    fault_paths={
        "liquidpipe01bar": "../data/raw/RTU_sim_liquidpipe01bar.csv",
        "liquidpipe04bar": "../data/raw/RTU_sim_liquidpipe04bar.csv",
        "liquidpipe08bar": "../data/raw/RTU_sim_liquidpipe08bar.csv",
        "liquidpipe10bar": "../data/raw/RTU_sim_liquidpipe10bar.csv",
    },
    pressure_temp_cols=("RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP", "RTU_REFG_DISC_PRES"),
)

print(f"Feature table shape: {table.shape}")
print(f"\nLabel distribution:\n{table['label'].value_counts()}")
table.head()

Feature table shape: (289382, 7)

Label distribution:
label
1    226192
0     63190
Name: count, dtype: int64


,Datetime,label,source_file,RTU_REFG_SUCT_PRES_residual,RTU_REFG_SUCT_TEMP_residual,RTU_REFG_DISC_PRES_residual,RTU_TOT_CAPA_ewma30_segmented_residual
0,2018-07-20 01:00:00,0,baseline,-9972.297657,0.704927,381038.683338,662.149504
1,2018-07-20 01:00:00,1,liquidpipe01bar,6953.702343,0.777105,869990.683338,579.692504
2,2018-07-20 01:00:00,1,liquidpipe08bar,-897883.297657,15.495706,337284.683338,-471.006496
3,2018-07-20 01:00:00,1,liquidpipe04bar,-7059.297657,0.634172,915192.683338,664.741504
4,2018-07-20 01:01:00,1,liquidpipe04bar,137082.889354,2.324666,998052.907863,750.799053


## Evaluating liquid-line restriction: both random-split and TimeSeriesSplit

In [2]:
feature_cols = [
    "RTU_REFG_SUCT_PRES_residual",
    "RTU_REFG_SUCT_TEMP_residual",
    "RTU_REFG_DISC_PRES_residual",
    "RTU_TOT_CAPA_ewma30_segmented_residual",
]

X_all = table[feature_cols].values
y_all = table["label"].values

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_random.fit(X_tr_r, y_tr_r)
y_pred_r = rf_random.predict(X_te_r)

print("=== Random split ===")
print(classification_report(y_te_r, y_pred_r, target_names=["baseline", "liquidline"]))

tscv = TimeSeriesSplit(n_splits=5)
print("=== TimeSeriesSplit (5 folds) ===")
for fold_num, (train_idx, test_idx) in enumerate(tscv.split(X_all), start=1):
    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    fold_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    fold_model.fit(X_tr, y_tr)
    y_pred_fold = fold_model.predict(X_te)

    report = classification_report(y_te, y_pred_fold, target_names=["baseline", "liquidline"], output_dict=True)
    print(f"Fold {fold_num}: baseline recall={report['baseline']['recall']:.2f}, "
          f"baseline precision={report['baseline']['precision']:.2f}, "
          f"liquidline recall={report['liquidline']['recall']:.2f}")

=== Random split ===
              precision    recall  f1-score   support

    baseline       0.99      0.98      0.99     12638
  liquidline       1.00      1.00      1.00     45239

    accuracy                           0.99     57877
   macro avg       0.99      0.99      0.99     57877
weighted avg       0.99      0.99      0.99     57877

=== TimeSeriesSplit (5 folds) ===
Fold 1: baseline recall=0.99, baseline precision=0.98, liquidline recall=0.99
Fold 2: baseline recall=1.00, baseline precision=0.97, liquidline recall=0.99
Fold 3: baseline recall=0.99, baseline precision=0.97, liquidline recall=0.99
Fold 4: baseline recall=1.00, baseline precision=0.96, liquidline recall=0.99
Fold 5: baseline recall=1.00, baseline precision=0.96, liquidline recall=0.99


## Liquid-line restriction: stable, near-perfect — matches condenser fouling's
## pattern despite the threshold-shaped severity response

| | Random split | TS Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|---|---|---|---|---|---|---|
| Baseline recall | 0.98 | 0.99 | 1.00 | 0.99 | 1.00 | 1.00 |
| Baseline precision | 0.99 | 0.98 | 0.97 | 0.97 | 0.96 | 0.96 |

No degradation trend, near-perfect throughout - matching condenser fouling's stable
pattern, not undercharge's collapse or evaporator fouling's gradual decline. This is
genuinely interesting given this fault's threshold-shaped severity response
(minimal change at 1-4 bar per the EDA) - the model still generalizes cleanly
forward-in-time despite half the fault-labeled data being only mildly different from
baseline. Suggests threshold-shaped vs. continuously-scaling severity is NOT the
distinguishing factor behind the three generalization patterns seen so far - this
fault is threshold-shaped yet behaves like the "stable" faults, not creating a
fourth pattern.

**Updated status on the open question**: 4 of 6 faults now modeled. Stable
(overcharge, condenser fouling, liquid-line restriction) vs. degrading (undercharge,
evaporator fouling) is now a 3-vs-2 split. No obvious shared property yet identifies
which faults land in which group - not signal strength alone (evaporator fouling has
the strongest signal yet degrades), not severity shape (liquid-line restriction has
a threshold shape yet is stable). Genuinely still an open question, one fault
remaining (suction-line restriction) to add one more data point before a real pattern
might emerge.

## Summary: liquid-line restriction binary classifier

Stable, near-perfect result (baseline recall 0.98-1.00 across every evaluation
method, no TimeSeriesSplit degradation) - matching condenser fouling's pattern.
Notably, this fault has a genuine threshold-shaped severity response (per notebook
05's EDA) yet still generalizes cleanly, suggesting severity shape is not what
determines the stable-vs-degrading split observed so far. Strongest, most stable
result of the five faults modeled to date.